# Geometry and Mesh Definition of Linear Actuator

We define the geometry and mesh consisting of a ferromagnetic core, a mover, a coil and a permanent magnet. We use OpenCascade to define the geometry and GMSH to generate the mesh. 

<b>Exercises</b>
1. document subdomain and edge labels in more details;
2. use filed_2d to round the inner edges of the core;
3. generate geometry and mesh for various mover positions; 

## Import Packages 

In [2]:
try
    using Gmsh: gmsh, Gmsh 
catch
    using gmsh
end 

## Section 1: OpenCascade - GMSH for Definition of Geometry - Mesh of Linear Actuator 

In [21]:
#..initialize GMSH 
gmsh.initialize()

gmsh.option.set_number("General.Verbosity", 2)
dim = 2;
zc = 0 # z-coordinate 

#..step-1: create and name core domain.. 
out_tag  = gmsh.model.occ.add_rectangle(-0.05, 0, zc, 0.1, 0.0405)
inn_tag  = gmsh.model.occ.add_rectangle(-0.04, 0.01, zc, .08, .0205)
gap_tag  = gmsh.model.occ.add_rectangle(-0.021, 0.0305, zc, .042, .01)
core_tag = gmsh.model.occ.cut([(2, out_tag)], [(2, inn_tag)])
core_tag = gmsh.model.occ.cut([(2, out_tag)], [(2, gap_tag)])
gmsh.model.setEntityName(dim,core_tag[1][1][2],"core")

#..step-2: create and name mover domain.. 
mover_tag = gmsh.model.occ.add_rectangle(-0.020, 0.0305, zc, .04, .01)
gmsh.model.setEntityName(dim,mover_tag,"mover")

#..step-3: create and name permanent magnet domain.. 
pm_tag    = gmsh.model.occ.add_rectangle(-0.010, 0.01, zc, .02, .02)
gmsh.model.setEntityName(dim,pm_tag,"pm")

#..step-4: create and name left coil up and left coil down.. 
left_coil_up_tag  = gmsh.model.occ.add_rectangle(-0.04, 0.01, zc, 0.03, 0.01)
gmsh.model.setEntityName(dim,left_coil_up_tag,"left-coil-up")
left_coil_dw_tag  = gmsh.model.occ.copy([(2, left_coil_up_tag)])
gmsh.model.occ.translate(left_coil_dw_tag, 0, -0.02, 0)
gmsh.model.setEntityName(dim,left_coil_dw_tag[1][2],"left-coil-dw")

#..step-5: create and name right coil up and right coil down.. 
right_coil_up_tag  = gmsh.model.occ.add_rectangle(0.01, 0.01, zc, 0.03, 0.01)
gmsh.model.setEntityName(dim,right_coil_up_tag,"right-coil-up")
right_coil_dw_tag  = gmsh.model.occ.copy([(2, right_coil_up_tag)])
gmsh.model.occ.translate(right_coil_dw_tag, 0, -0.02, 0)
gmsh.model.setEntityName(dim,right_coil_dw_tag[1][2],"right-coil-dw")

#..step-6: create and name air domain 
air_tag = gmsh.model.occ.add_rectangle(-0.08, -0.03, zc, 0.16, 0.09)
gmsh.model.setEntityName(dim,air_tag,"air")

#..step-7: synchronize  model (required here in order for model.getEntities() to work properly) 
gmsh.model.occ.synchronize()

#..step-8: get entities  
entities = gmsh.model.getEntities() # see documentation of getEnties 

#..step-9: subtract core from air domain 
for e in entities
    dim  = e[1]
    tag  = e[2]
    name = gmsh.model.getEntityName(dim,tag)

    if ((name=="core")||(name=="mover")||(name=="pm")||(name=="left-coil-up")||(name=="left-coil-dw")||(name=="right-coil-up")||(name=="right-coil-dw"))
        copy = gmsh.model.occ.copy(e)
        difference = gmsh.model.occ.cut([(dim, air_tag)],e) 
    end 
    
end

#..step-10: synchronize  model
gmsh.model.occ.synchronize()

#..step-11: assign physical groups for the boundaries and subdomains 
#..step-11-a: physical groups for the eight subdomains 
airtag           = gmsh.model.add_physical_group(2, [8], -1, "air")
coretag          = gmsh.model.add_physical_group(2, [9], -1, "core")
movertag         = gmsh.model.add_physical_group(2, [10], -1, "mover")
magnettag        = gmsh.model.add_physical_group(2, [11], -1, "magnet")
left_coil_uptag  = gmsh.model.addPhysicalGroup(2, [12], -1, "left_coil_up")
left_coil_dwtag  = gmsh.model.addPhysicalGroup(2, [13], -1, "left_coil_dw")
right_coil_uptag = gmsh.model.addPhysicalGroup(2, [14], -1, "right_coil_up")
right_coil_dwtag = gmsh.model.addPhysicalGroup(2, [15], -1, "right_coil_dw")
#..step-11-b: physical groups for the four boundaries
bottomtag = gmsh.model.add_physical_group(1, [203], -1, "bottom")
lefttag   = gmsh.model.add_physical_group(1, [204], -1, "left")
righttag  = gmsh.model.add_physical_group(1, [205], -1, "right")
toptag    = gmsh.model.add_physical_group(1, [206], -1, "top")

#..step-1: synchronize  model (required here in order for mesh generation to work properly)
gmsh.model.occ.synchronize()

gmsh.option.setNumber("Mesh.MeshSizeMax",0.0005)

gmsh.model.mesh.generate(dim)
gmsh.model.mesh.removeDuplicateNodes()
gmsh.model.mesh.renumberNodes()

#..if true, write mesh to file for further processing
if (true) gmsh.write("linear-actuator.msh") end 

#..if true, visualize mesh through the GUI 
if (true) gmsh.fltk.run() end 

#..finalize GMSH
gmsh.finalize()